# ONIA — 5.10 Clasificare și regresie pe texte

Exemple educaționale: calcule de la zero și API scikit-learn. Toate datele sunt sintetice. Pentru a executa exemplele este necesar `scikit-learn`.

## Exemplul 1 — Definirea clară a țintei

Un exemplu în care fiecare document are exact o etichetă. Este clasificare multiclasă.

In [ ]:
texte = ["sport fotbal meci", "algoritm python cod", "știre despre educație"]
etichete = ["sport", "tehnologie", "educație"]
for i, (text, eticheta) in enumerate(zip(texte, etichete), start=1):
    print(i, "|", eticheta, "|", text)


## Exemplul 2 — Construirea vocabularului de la zero

Fiecare linie este un document, fiecare coloană este un termen. Multe valori sunt zero.

In [ ]:
from collections import Counter
texte = ["tenis sport meci", "sport fotbal stadion", "python cod algoritm"]
vocabular = sorted(set(" ".join(texte).split()))
print("Vocabular:", vocabular)
for text in texte:
    frecv = Counter(text.split())
    print([frecv[cuvant] for cuvant in vocabular])


## Exemplul 3 — Multinomial Naive Bayes from scratch

Se pot verifica manual priorii, numărătorile și efectul lui alpha.

In [ ]:
from collections import Counter, defaultdict
from math import log

texte = ["sport tenis", "sport fotbal", "python algoritm", "cod python"]
y = ["sport", "sport", "tech", "tech"]
clase = sorted(set(y))
vocab = set(" ".join(texte).split())
numere = {c: Counter() for c in clase}
totaluri = {c: 0 for c in clase}
prior = Counter(y)
for text, clasa in zip(texte, y):
    cuvinte = text.split()
    numere[clasa].update(cuvinte)
    totaluri[clasa] += len(cuvinte)

alpha = 1.0
nou = "tenis sport"
scoruri = {}
for clasa in clase:
    scor = log(prior[clasa] / len(y))
    for cuvant in nou.split():
        if cuvant in vocab:
            p = (numere[clasa][cuvant] + alpha) / (totaluri[clasa] + alpha * len(vocab))
            scor += log(p)
    scoruri[clasa] = scor
print("Scoruri log:", {c: round(v, 3) for c,v in scoruri.items()})
print("Predicție:", max(scoruri, key=scoruri.get))


## Exemplul 4 — Cât schimbă smoothingul?

Termenul absent primește probabilitate nenulă pentru alpha pozitiv.

In [ ]:
numar_termen = 0
numar_total = 8
vocab = 4
for alpha in (0.1, 0.5, 1.0, 2.0):
    p = (numar_termen + alpha) / (numar_total + alpha * vocab)
    print(f"alpha={alpha:<4} P={p:.4f}")


## Exemplul 5 — MultinomialNB versus BernoulliNB

Comparăm aceleași texte cu modele care pun accent pe numărări versus prezență.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB

X = ["tenis sport", "sport meci", "fotbal sport", "python cod", "cod algoritm", "date python"]
y = [0, 0, 0, 1, 1, 1]
for model in (MultinomialNB(alpha=1.0), BernoulliNB(alpha=1.0)):
    pipe = make_pipeline(CountVectorizer(binary=isinstance(model, BernoulliNB)), model)
    pipe.fit(X, y)
    print(type(model).__name__, pipe.predict(["sport tenis", "algoritm python"]))


## Exemplul 6 — ComplementNB pentru clase rare

Experiment didactic: nu interpretează ca rezultat validat pe clasa rară.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB

X = ["meci sport", "sport stadion", "sport fotbal", "tenis sport", "sport echipă",
     "python algoritm", "algoritm cod"]
y = ["sport"] * 5 + ["tech"] * 2
model = make_pipeline(TfidfVectorizer(), ComplementNB(alpha=0.5))
model.fit(X, y)
print(model.predict(["cod algoritm", "sport meci"]))


## Exemplul 7 — O iterație de regresie logistică de la zero

Vezi cum modelul crește probabilitatea clasei pozitive pentru un exemplu pozitiv.

In [ ]:
from math import exp

x, y = 2.0, 1.0
w, b, lr = 0.0, 0.0, 0.1
for pas in range(5):
    z = w*x + b
    p = 1 / (1 + exp(-z))
    dw, db = (p - y)*x, (p - y)
    print(pas, f"p={p:.3f} dw={dw:.3f} db={db:.3f}")
    w -= lr*dw
    b -= lr*db


## Exemplul 8 — LogisticRegression și predict_proba

C este invers legat de intensitatea regularizării. Probabilitățile sunt estimări, nu garanții.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X = ["sport tenis", "meci fotbal", "echipa stadion", "python cod", "algoritm date", "programare python"]
y = ["sport"]*3 + ["tech"]*3
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(C=1.0, max_iter=1000))
model.fit(X, y)
print("Clase:", model[-1].classes_)
print("Probabilități:", model.predict_proba(["tenis stadion"]).round(3))


## Exemplul 9 — LinearSVC pentru text sparse

decision_function nu produce probabilitate.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

X = ["tenis sport", "fotbal stadion", "finala echipă", "cod python", "algoritm recursiv", "date programare"]
y = ["sport"]*3 + ["tech"]*3
model = Pipeline([("tfidf", TfidfVectorizer()), ("svc", LinearSVC(C=0.5, random_state=0))])
model.fit(X, y)
print("Predicție:", model.predict(["meci tenis"]))
print("Scor față de frontieră:", model.decision_function(["meci tenis"]))


## Exemplul 10 — Comparație cu același split

Datele sunt foarte mici; scopul este folosirea corectă a aceluiași split, nu un clasament al algoritmilor.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

X = ["tenis sport", "fotbal sport", "meci stadion", "gol echipă",
     "python cod", "algoritm date", "programare cod", "baze de date"]
y = ["sport"]*4 + ["tech"]*4
xt, xv, yt, yv = train_test_split(X, y, test_size=0.5, random_state=2, stratify=y)
for estimator in (MultinomialNB(alpha=1.0), LinearSVC(C=1.0)):
    model = make_pipeline(TfidfVectorizer(), estimator)
    model.fit(xt, yt)
    pred = model.predict(xv)
    print(type(estimator).__name__, f1_score(yv, pred, average="macro", zero_division=0))


## Exemplul 11 — Multi-class cu trei clase

Un document primește o singură clasă dintre cele trei.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X = ["tenis meci sport", "fotbal sport echipă", "programare python cod",
     "cod algoritm date", "educație școală elevi", "profesor elevi manual"]
y = ["sport", "sport", "tech", "tech", "educație", "educație"]
m = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
m.fit(X, y)
print(m.predict(["meci de fotbal", "lecție pentru elevi"]))
print(m[-1].classes_)


## Exemplul 12 — Multilabel cu OneVsRestClassifier

Etichetele multilabel sunt o matrice binară, nu un singur șir.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

X = ["model inteligenta artificiala", "elev scoala invatare", "ai la scoala",
     "meci fotbal", "algoritm ai", "manual scoala"]
y = [["IA"], ["educatie"], ["IA", "educatie"], ["sport"], ["IA"], ["educatie"]]
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(y)
pipe = Pipeline([("tfidf", TfidfVectorizer()),
                 ("clf", OneVsRestClassifier(LogisticRegression(max_iter=1000)))])
pipe.fit(X, Y)
print("Etichete:", mlb.classes_)
print("Predicție:", mlb.inverse_transform(pipe.predict(["ai pentru scoala"])))


## Exemplul 13 — Ridge: regresie numerică din text

Scorurile sunt ilustrative; nu reprezintă un benchmark ONIA.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

X = ["problema simpla suma", "sir simplu", "cautare element", "graf drum minim",
     "programare dinamica complexa", "arbore segment interval"]
y = [1.0, 1.4, 2.0, 4.0, 4.7, 4.2]
m = make_pipeline(TfidfVectorizer(), Ridge(alpha=1.0))
m.fit(X, y)
print("Dificultate estimată:", m.predict(["drum minim in graf"]).round(2))


## Exemplul 14 — MAE și RMSE calculate de la zero

RMSE penalizează mai puternic erorile mari.

In [ ]:
from math import sqrt
real = [2, 3, 4, 5]
pred = [2.2, 2.5, 4.3, 4.6]
mae = sum(abs(a-b) for a,b in zip(real, pred)) / len(real)
rmse = sqrt(sum((a-b)**2 for a,b in zip(real, pred)) / len(real))
print(f"MAE={mae:.3f} RMSE={rmse:.3f}")


## Exemplul 15 — SGDClassifier cu loss și regularizare

loss schimbă obiectivul optimizării; alpha regularizează.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

X = ["sport fotbal", "tenis sport", "echipă meci", "python cod", "cod algoritm", "date python"]
y = [0, 0, 0, 1, 1, 1]
for loss in ("hinge", "log_loss"):
    m = make_pipeline(TfidfVectorizer(), SGDClassifier(loss=loss, alpha=0.001, random_state=4, max_iter=2000))
    m.fit(X, y)
    print(loss, m.predict(["meci tenis"]))


## Exemplul 16 — Căutare pe pipeline cu GridSearchCV

Vectorizarea se învață separat în fiecare fold, nu o singură dată pe tot corpusul.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

X = ["tenis sport", "sport meci", "stadion fotbal", "fotbal echipă", "tenis finală", "echipă sport",
     "python cod", "algoritm date", "cod recursiv", "python liste", "algoritm graf", "programare date"]
y = ["sport"]*6 + ["tech"]*6
pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))])
parametri = {"tfidf__ngram_range": [(1, 1), (1, 2)], "clf__C": [0.1, 1.0, 10.0]}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
search = GridSearchCV(pipe, parametri, cv=cv, scoring="f1_macro")
search.fit(X, y)
print(search.best_params_)
print("CV macro-F1:", round(search.best_score_, 3))


## Exemplul 17 — Coeficienți și termeni pentru două clase

Coeficienții sunt descriptivi pentru modelul antrenat, nu cauze ale etichetelor.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X = ["tenis meci sport", "echipă sport stadion", "finala fotbal sport",
     "python cod algoritm", "algoritm date python", "programare cod date"]
y = ["sport"]*3 + ["tech"]*3
m = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
m.fit(X, y)
term = m[0].get_feature_names_out()
coef = m[1].coef_[0]
print("Clasa pozitivă:", m[1].classes_[1])
print([(term[i], round(float(coef[i]), 3)) for i in coef.argsort()[-4:][::-1]])


## Exemplul 18 — Matrix de confuzie și F1 macro

Matricea se interpretează pe clase, nu numai prin diagonală.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
real = ["sport", "sport", "tech", "tech", "educatie", "educatie"]
pred = ["sport", "tech", "tech", "tech", "sport", "educatie"]
clase = ["sport", "tech", "educatie"]
print(confusion_matrix(real, pred, labels=clase))
print("Macro-F1:", round(f1_score(real, pred, average="macro"), 3))
print(classification_report(real, pred, labels=clase, zero_division=0))


## Exemplul 19 — Separarea grupurilor în evaluare

Versiunile aceleiași probleme nu apar simultan în train și validation.

In [ ]:
from sklearn.model_selection import GroupKFold
texte = ["var A sport", "var B sport", "date python", "cod python", "meci final", "sport final"]
y = ["sport", "sport", "tech", "tech", "sport", "sport"]
grupuri = ["problema1", "problema1", "problema2", "problema2", "problema3", "problema3"]
cv = GroupKFold(n_splits=3)
for train, val in cv.split(texte, y, groups=grupuri):
    print("Grupuri train:", set(grupuri[i] for i in train),
          "| validation:", set(grupuri[i] for i in val))


## Exemplul 20 — Predictii pentru o problemă de concurs

Date complet sintetice: verifică formatul exact cerut de concurs înaintea unei trimiteri.

In [ ]:
import csv
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

texte_train = ["graf drum", "arbore parcurgere", "numar prim divizor", "factorizare prime",
               "sir palindrom", "subsir text", "puncte geometrie", "arie triunghi",
               "programare dinamica", "subproblema dp"]
y_train = ["graphs"]*2 + ["number theory"]*2 + ["strings"]*2 + ["geometry"]*2 + ["dp"]*2
m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LinearSVC(C=1.0))
m.fit(texte_train, y_train)
id_test = [101, 102]
text_test = ["drum intre noduri", "sir caractere"]
predictii = m.predict(text_test)
with open("predictii_demo.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "label"])
    w.writerows(zip(id_test, predictii))
print(list(zip(id_test, predictii)))


## Exerciții propuse

1. Comparați Macro-F1 pentru două clasificatoare pe aceeași validare.
2. Modificați `ngram_range`, apoi interpretați cazurile în care rezultatele se schimbă.
3. Construiți un set cu clase rare și evaluați efectul `class_weight`.
4. Comparați MAE și RMSE pentru aceeași regresie.